# 🌍 Groq Multilingual Voice Chatbot

Speak naturally in a supported language. Whisper automatically detects the spoken language, the LLM replies in that same language, and gTTS speaks the response.

**Pipeline:** 🎤 Microphone → Whisper Large v3 → Language detection → Multilingual LLM → gTTS → 🔊 Voice

## 1. Install dependencies

In [1]:
!pip -q install -U groq gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.


## 2. Add your Groq API keys to Colab Secrets

In the 🔑 **Secrets** panel add `GROQ_API_KEY_STT` and `GROQ_API_KEY_LLM`.

In [3]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY_STT = userdata.get("GROQ_API_KEY_STT")
GROQ_API_KEY_LLM = userdata.get("GROQ_API_KEY_LLM")

if not GROQ_API_KEY_STT or not GROQ_API_KEY_LLM:
    raise ValueError("Add GROQ_API_KEY_STT and GROQ_API_KEY_LLM to Colab Secrets.")

stt_client = Groq(api_key=GROQ_API_KEY_STT)
llm_client = Groq(api_key=GROQ_API_KEY_LLM)

STT_MODEL = "whisper-large-v3"
LLM_MODEL = "openai/gpt-oss-20b"
print("Ready!")

Ready!


## 3. Browser microphone recorder

No file upload is required. Your browser will request microphone permission.

In [4]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def record_audio(seconds=8, filename="recording.webm"):
    display(Javascript(r'''
    async function recordAudio(seconds) {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, seconds * 1000));
      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      const blob = new Blob(chunks, {type: 'audio/webm'});
      const reader = new FileReader();
      return await new Promise(resolve => {
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
      });
    }
    '''))
    data = eval_js(f"recordAudio({seconds})")
    audio = b64decode(data.split(",")[1])
    with open(filename, "wb") as f:
        f.write(audio)
    return filename

## 4. TTS language mapping

In [19]:
# Map language names/codes → gTTS language codes

TTS_LANGUAGE_MAP = {
    # Indian languages
    "hindi": "hi",
    "hi": "hi",

    "marathi": "mr",
    "mr": "mr",

    "urdu": "ur",
    "ur": "ur",

    "bengali": "bn",
    "bangla": "bn",
    "bn": "bn",

    "gujarati": "gu",
    "gu": "gu",

    "tamil": "ta",
    "ta": "ta",

    "telugu": "te",
    "te": "te",

    "kannada": "kn",
    "kn": "kn",

    "malayalam": "ml",
    "ml": "ml",

    "punjabi": "pa",
    "pa": "pa",

    "nepali": "ne",
    "ne": "ne",

    "english": "en",
    "en": "en",
}


def get_tts_language(detected_language):
    """
    Convert Whisper's detected language into
    the language code expected by gTTS.
    """

    if detected_language is None:
        return "en"

    language = str(detected_language).strip().lower()

    return TTS_LANGUAGE_MAP.get(language, "en")

## 5. 🎤 Speak → detect → respond → 🔊 speak

Run this cell for each turn. You do **not** need to specify the language.

In [23]:
import time
from gtts import gTTS
from IPython.display import Audio, display

RECORD_SECONDS = 4

print(f"🎤 Speak now ({RECORD_SECONDS} seconds)...")
audio_path = record_audio(RECORD_SECONDS)

# Whisper: detect language + transcribe
with open(audio_path, "rb") as f:
    start = time.perf_counter()
    transcription = stt_client.audio.transcriptions.create(
        file=(audio_path, f.read()),
        model=STT_MODEL,
        response_format="verbose_json",
        temperature=0.0,
    )
    stt_time = time.perf_counter() - start

transcript = transcription.text.strip()
detected_language = getattr(transcription, "language", None) or "en"
tts_language = get_tts_language(detected_language)

print(f"🌍 Detected language: {detected_language}")
print(f"📝 You: {transcript}")
print(f"⏱️ STT: {stt_time:.2f}s")

# LLM: force response in the same language
start = time.perf_counter()
response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role":"system","content":
         "You are a friendly multilingual voice chatbot. Reply in exactly the same language as the user's message. "
         "Do not translate into another language. Use natural conversational language and keep replies concise."},
        {"role":"user","content":transcript}
    ],
    reasoning_effort="medium",
    temperature=0.7,
    max_completion_tokens=500,
)
llm_time = time.perf_counter() - start
answer = response.choices[0].message.content.strip()

print(f"\n🤖 Bot: {answer}")
print(f"⏱️ LLM: {llm_time:.2f}s")

# TTS: same detected language
print(f"\n🔊 Speaking in: {tts_language}")
gTTS(text=answer, lang=tts_language).save("bot_response.mp3")
display(Audio("bot_response.mp3", autoplay=True))

🎤 Speak now (4 seconds)...


<IPython.core.display.Javascript object>

🌍 Detected language: Indonesian
📝 You: Tommy Kim Choo
⏱️ STT: 0.39s

🤖 Bot: Hi Tommy! How can I help you today?
⏱️ LLM: 0.26s

🔊 Speaking in: en


## 6. Test languages

Try English, Hindi, Marathi, Urdu, Tamil, Telugu, Bengali, Gujarati, Kannada, Malayalam, or Punjabi. The language is selected automatically from Whisper's detection result.

**Next upgrade:** add persistent conversation history and a Record/Stop button for natural back-and-forth conversation.